<a href="https://colab.research.google.com/github/Guliko24/PubMed_Fetch/blob/main/Week2_Day3_Structured_Extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 Day 3: Structured extraction with Gemini

Deliverable: one abstract yields schema-valid claims, and missing items are not guessed.

`src/pubmed_fetch/extract.py` sends one abstract to Gemini with structured output (`ExtractedClaims`), then attaches the PMID itself and returns an `AbstractExtraction`. The Day 2 quote check then tests every `supporting_quote`.

Before running: add your Gemini key in Colab **Secrets** as `GOOGLE_API_KEY` and allow this notebook to access it.

In [ ]:
from pathlib import Path
import subprocess
import sys

GITHUB_USERNAME = "Guliko24"
REPO_NAME = "PubMed_Fetch"
BRANCH = "main"  # set to a branch name to test it before merging

repo_dir = Path("/content") / REPO_NAME
repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

def git(*args):
    return subprocess.run(
        ["git", "-C", str(repo_dir), *args], check=True, capture_output=True, text=True
    ).stdout.strip()

if not repo_dir.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, repo_url, str(repo_dir)], check=True)
else:
    # /content survives "Restart session", so an older clone may be on another branch.
    git("fetch", "origin", BRANCH)
    git("checkout", BRANCH)
    git("pull", "--ff-only", "origin", BRANCH)

print(f"Clone at {repo_dir} is on branch {git('branch', '--show-current')}, commit {git('rev-parse', '--short', 'HEAD')}")

%pip -q install -r {repo_dir}/requirements.txt

# Until pyproject.toml arrives in Week 5, put src/ on the import path by hand.
for path in (repo_dir, repo_dir / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import search_utils
from pubmed_fetch import extract, graph_basics, schemas
print("Package imported from:", extract.__file__)

In [ ]:
import os

# Colab Secrets are not environment variables: copy the key into one,
# so the same code also works with a .env file on a laptop.
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass  # not in Colab: extract.get_llm() reads .env instead

# Never print the key itself: notebook outputs are pushed to a public repo.
print("GOOGLE_API_KEY set:", bool(os.environ.get("GOOGLE_API_KEY")))

## 1. Choose a model

Model names change often, so this asks your key which Flash models it can use. The next cell uses `gemini-3.6-flash`, the model of the Day 3 run. If it is retired, copy a listed name (without the `models/` prefix) instead; prefer one not marked `preview`.

In [ ]:
from google import genai

client = genai.Client()  # reads GOOGLE_API_KEY from the environment
for model in client.models.list():
    if "flash" in model.name and "generateContent" in (model.supported_actions or []):
        print(model.name)

In [ ]:
GEMINI_MODEL = "gemini-3.6-flash"  # used for the Day 3 run; replace with a listed name if retired

assert GEMINI_MODEL, "Set GEMINI_MODEL to one of the names listed above."
os.environ["GEMINI_MODEL"] = GEMINI_MODEL
llm = extract.get_llm()
print("Using model:", GEMINI_MODEL)

## 2. Extract claims from one abstract

In [ ]:
documents = search_utils.load_pubmed_data(repo_dir / "pubmed_abstracts.json")
abstracts = {doc["pmid"]: doc["abstract"] for doc in documents}


def show(extraction, abstract):
    """Print each claim with the Day 2 literal quote check."""
    print(f"PMID {extraction.pmid}: {len(extraction.claims)} claims\n")
    for number, claim in enumerate(extraction.claims, start=1):
        found = graph_basics.quote_in_abstract(claim.supporting_quote, abstract)
        print(f"[{number}] quote found: {found}")
        print(f"    cell_type:    {claim.cell_type}")
        print(f"    marker_genes: {claim.marker_genes}")
        print(f"    disease:      {claim.disease}")
        print(f"    claim_text:   {claim.claim_text}")
        print(f"    quote:        {claim.supporting_quote}\n")


PMID = "34582785"
result = extract.extract_claims(PMID, abstracts[PMID], llm=llm)
show(result, abstracts[PMID])

## 3. An abstract with HTML markup

This abstract contains tags such as `<b>Rationale:</b>`. The instructions say to copy quotes *including any markup*, so the literal quote check should still find them.

In [ ]:
PMID_MARKUP = "40585969"
result_markup = extract.extract_claims(PMID_MARKUP, abstracts[PMID_MARKUP], llm=llm)
show(result_markup, abstracts[PMID_MARKUP])

## 4. Missing items are not guessed

A made-up text that names no cell type. The correct answer is an empty `claims` list, not invented claims.

In [ ]:
no_cell_types = (
    "This review summarises statistical methods for analysing sequencing data "
    "and discusses open problems in study design."
)
result_empty = extract.extract_claims("test-0", no_cell_types, llm=llm)
print("Claims returned:", len(result_empty.claims), "(expected 0)")

## What to check

- Every call returns without a validation error: the output is schema-valid.
- Section 2: claims name MSNs or interneurons; `disease` is `null` unless a claim's evidence names one; every `quote found` is `True`. Any `False` is a quote the model did not copy exactly. Note it for Day 5.
- Section 3: note whether quotes that cross a `<b>` tag were copied with the markup.
- Section 4: `0` claims.
- LLM output can vary between runs even at temperature 0. Record what you saw; do not treat one run as a measured accuracy.